In [0]:
%sql
create table if not exists cyntexa_dev.sales.unmasked_access_list (
    user_email string
);

In [0]:
%sql
insert into cyntexa_dev.sales.unmasked_access_list values
('rinkimehrasalesforce@gmail.com')

In [0]:
%sql
create or replace function cyntexa_dev.sales.masked_values(inputStr STRING)
returns string
return case
    when exists (
        select 1 from cyntexa_dev.sales.unmasked_access_list
        where user_email = current_user()
    ) then inputstr
    when inputStr is null then null
    when length(inputStr) <= 4 then repeat('*', length(inputStr))
    when inputStr like '%@%' then
        concat(
            substring(split(inputStr, '@')[0], 1, length(split(inputStr, '@')[0]) - 4),
            '****',
            '@',
            split(inputStr, '@')[1]
        )
    else concat(substring(inputStr, 1, length(inputStr) - 4), '****')
end

In [0]:
%sql
alter table cyntexa_dev.sales.orders_raw
alter column email
set mask cyntexa_dev.sales.masked_values;

In [0]:
%sql
select customer_id, email from cyntexa_dev.sales.orders_raw;